# MDict 词典元数据分析

读取一个 MDX 文件及其配套 MDD，统计词条、资源和 MDX 内容中的音频引用。分析只读取数据，不会提取或修改词典文件。


In [1]:
from collections import Counter
from pathlib import Path
import re
from urllib.parse import unquote

from mdict_utils.readmdict import MDX, MDD

# 修改为待分析的 MDX 路径；同名 .mdd 会被自动发现。
MDX_PATH = Path(r"/data/dicts/longman6/LongmanDictionaryOfContemporaryEnglish6thEnEn.mdx")
# 如 MDD 不与 MDX 同名，可指定路径；否则保持为 None。
MDD_PATH = None

AUDIO_EXTENSIONS = {".mp3", ".wav", ".ogg", ".oga", ".spx", ".opus", ".m4a", ".aac", ".flac"}


LZO compression support is not available


In [2]:
SOUND_SCHEME_RE = re.compile(r"sound://([^\"'<>\s]+)", re.IGNORECASE)
HTML_AUDIO_RE = re.compile(
    r"(?:src|href)\s*=\s*['\"](?!sound://)([^'\"]+\.(?:mp3|wav|ogg|oga|spx|opus|m4a|aac|flac)(?:[?#][^'\"]*)?)['\"]",
    re.IGNORECASE,
)


def to_text(value):
    """将 readmdict 返回的 bytes 安全转换为文本。"""
    if isinstance(value, bytes):
        return value.decode("utf-8", errors="replace")
    return str(value)


def normalize_resource_path(value):
    """统一 URL 编码、斜杠、查询参数和大小写，便于 MDX/MDD 路径匹配。"""
    path = unquote(to_text(value)).strip().replace("\\", "/")
    path = path.split("?", 1)[0].split("#", 1)[0]
    return path.lstrip("/").casefold()


def extract_audio_references(record):
    """提取 MDX 记录中的 sound:// 及 HTML 音频链接，保留重复引用。"""
    text = to_text(record)
    references = [match.group(1) for match in SOUND_SCHEME_RE.finditer(text)]
    references.extend(match.group(1) for match in HTML_AUDIO_RE.finditer(text))
    return [normalize_resource_path(ref) for ref in references if normalize_resource_path(ref)]


def decode_header(header):
    return {to_text(key): to_text(value) for key, value in header.items()}


def analyze_dictionary(mdx_path, mdd_path=None, sample_limit=20):
    """扫描词典并返回可序列化的元数据字典。"""
    mdx_path = Path(mdx_path).expanduser().resolve()
    if not mdx_path.is_file() or mdx_path.suffix.casefold() != ".mdx":
        raise FileNotFoundError(f"MDX 文件不存在或扩展名不正确：{mdx_path}")

    if mdd_path is None:
        candidate = mdx_path.with_suffix(".mdd")
        mdd_path = candidate if candidate.is_file() else None
    else:
        mdd_path = Path(mdd_path).expanduser().resolve()
        if not mdd_path.is_file() or mdd_path.suffix.casefold() != ".mdd":
            raise FileNotFoundError(f"MDD 文件不存在或扩展名不正确：{mdd_path}")

    mdx = MDX(str(mdx_path))
    reference_counts = Counter()
    entries_with_audio = 0
    sample_entries = []

    for key, record in mdx.items():
        references = extract_audio_references(record)
        if not references:
            continue
        entries_with_audio += 1
        reference_counts.update(references)
        if len(sample_entries) < sample_limit:
            sample_entries.append({"entry": to_text(key), "audio": sorted(set(references))})

    mdd_metadata = None
    mdd_resources = set()
    audio_resources = []
    if mdd_path is not None:
        mdd = MDD(str(mdd_path))
        mdd_resources = {normalize_resource_path(key) for key in mdd.keys()}
        audio_resources = sorted(
            path for path in mdd_resources if Path(path).suffix.casefold() in AUDIO_EXTENSIONS
        )
        mdd_metadata = {
            "path": str(mdd_path),
            "size_bytes": mdd_path.stat().st_size,
            "resource_count": len(mdd),
            "header": decode_header(mdd.header),
            "audio_resource_count": len(audio_resources),
            "audio_format_counts": dict(sorted(Counter(Path(path).suffix for path in audio_resources).items())),
        }

    unique_references = set(reference_counts)
    matched = unique_references & mdd_resources if mdd_path is not None else set()
    missing = unique_references - mdd_resources if mdd_path is not None else unique_references

    return {
        "mdx": {
            "path": str(mdx_path),
            "size_bytes": mdx_path.stat().st_size,
            "entry_count": len(mdx),
            "header": decode_header(mdx.header),
        },
        "mdd": mdd_metadata,
        "audio": {
            "entries_with_audio_references": entries_with_audio,
            "reference_occurrences": sum(reference_counts.values()),
            "unique_references": len(unique_references),
            "matched_in_mdd": len(matched),
            "missing_from_mdd": len(missing),
            "matched_paths": sorted(matched),
            "missing_paths": sorted(missing),
            "sample_entries": sample_entries,
        },
    }


def print_summary(metadata):
    mdx = metadata["mdx"]
    mdd = metadata["mdd"]
    audio = metadata["audio"]
    print(f"MDX：{mdx['path']}")
    print(f"词条数：{mdx['entry_count']:,}，文件大小：{mdx['size_bytes']:,} bytes")
    if mdd:
        print(f"MDD：{mdd['path']}")
        print(f"资源数：{mdd['resource_count']:,}，音频资源数：{mdd['audio_resource_count']:,}")
        print(f"音频格式：{mdd['audio_format_counts']}")
    else:
        print("MDD：未找到配套文件，无法验证音频引用是否存在。")
    print(f"含音频引用的词条数：{audio['entries_with_audio_references']:,}")
    print(f"音频引用次数 / 唯一路径数：{audio['reference_occurrences']:,} / {audio['unique_references']:,}")
    if mdd:
        print(f"MDD 中已匹配 / 缺失：{audio['matched_in_mdd']:,} / {audio['missing_from_mdd']:,}")


In [4]:
metadata = analyze_dictionary(MDX_PATH, MDD_PATH)
print_summary(metadata)
metadata  # 在 Notebook 中展开完整头部、匹配路径、缺失路径和样例词条


MDX：/data/dicts/longman6/LongmanDictionaryOfContemporaryEnglish6thEnEn.mdx
词条数：52,749，文件大小：162,322,205 bytes
MDD：/data/dicts/longman6/LongmanDictionaryOfContemporaryEnglish6thEnEn.mdd
资源数：182,430，音频资源数：181,390
音频格式：{'.mp3': 181390}
含音频引用的词条数：52,590
音频引用次数 / 唯一路径数：276,173 / 181,429
MDD 中已匹配 / 缺失：181,387 / 42


{'mdx': {'path': '/data/dicts/longman6/LongmanDictionaryOfContemporaryEnglish6thEnEn.mdx',
  'size_bytes': 162322205,
  'entry_count': 52749,
  'header': {'GeneratedByEngineVersion': '2.0',
   'RequiredEngineVersion': '2.0',
   'Format': 'Html',
   'KeyCaseSensitive': 'No',
   'StripKey': 'Yes',
   'Encrypted': '2',
   'RegisterBy': 'EMail',
   'Description': 'faq-index',
   'Title': 'Longman Dictionary of Contemporary English 6th edition',
   'Encoding': 'UTF-8',
   'CreationDate': '2016-9-12',
   'Compact': 'No',
   'Compat': 'No',
   'Left2Right': 'Yes',
   'DataSourceFormat': '107',
   'StyleSheet': ''}},
 'mdd': {'path': '/data/dicts/longman6/LongmanDictionaryOfContemporaryEnglish6thEnEn.mdd',
  'size_bytes': 1298038335,
  'resource_count': 182430,
  'header': {'GeneratedByEngineVersion': '2.0',
   'RequiredEngineVersion': '2.0',
   'Format': '',
   'KeyCaseSensitive': 'No',
   'StripKey': 'No',
   'Encrypted': '2',
   'RegisterBy': 'EMail',
   'Description': 'faq-index',
   'Titl